# Module 12 lab · The maths inside a transformer

The code from every lesson of [this module](https://aicourses.dev/mathematics/inside-a-transformer), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/mathematics/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 12.2 Tokens and the embedding table

A token is a row number, and looking the row up is a one-hot vector times a matrix. This is the first box in the diagram and the first bit of maths from Module 03 you reuse.

[Read the lesson](https://aicourses.dev/mathematics/inside-a-transformer/embedding-table)

### In NumPy

The first block loads the course’s text and turns “alice was” into row numbers, as in Module 09. The second makes a small embedding table for the 46 characters, with 4 numbers in each row, and checks that the one-hot product and the lookup agree.

In [ ]:
import urllib.request
import numpy as np

rng = np.random.default_rng(0)
DATA = "https://raw.githubusercontent.com/aicoursesdev/mathematics-for-ai/main/data/"
urllib.request.urlretrieve(DATA + "alice.txt", "alice.txt")
text = " ".join(open("alice.txt", encoding="utf-8").read().lower().split())
chars = sorted(set(text))
V = len(chars)
ids = [chars.index(c) for c in "alice was"]
print(V, ids)
print([chars[i] for i in ids])

*The lesson shows:*

```text
46 [14, 25, 22, 16, 18, 0, 36, 14, 32]
['a', 'l', 'i', 'c', 'e', ' ', 'w', 'a', 's']
```

The space is token 0 and “a” is token 14, and the “a” that appears twice has the same row number both times. Now the hand example, then the real table:

In [ ]:
E = np.array([[0.5, -1, 0], [1, 0, 2], [0, 1, -1], [2, 1, 0], [-1, 0.5, 1]])
onehot = np.eye(5)[[3, 1, 3]]
print(onehot @ E)
print(E[[3, 1, 3]])

E = rng.normal(0, 4 ** -0.5, (V, 4))
onehot = np.eye(V)[ids]
print(onehot.shape, (onehot @ E).shape, np.allclose(onehot @ E, E[ids]))
print(46 * 4, 37000 * 512, round(512 ** 0.5, 4))

*The lesson shows:*

```text
[[2. 1. 0.]
 [1. 0. 2.]
 [2. 1. 0.]]
[[2. 1. 0.]
 [1. 0. 2.]
 [2. 1. 0.]]
(9, 46) (9, 4) True
184 18944000 22.6274
```

## 12.3 Where am I? Positions as sines and cosines

Each position gets a pattern of waves of different lengths, built from the sines and cosines of Module 01. You tabulate the first four positions by hand, then see how rotary positions differ.

[Read the lesson](https://aicourses.dev/mathematics/inside-a-transformer/positions)

### Why waves of different lengths

Is every position really different? The fast pair alone says no. Seven full turns of the circle are $7 \times 2\pi = 43.98$ radians, so after 44 steps the fast hand has almost returned to where it began, and positions 0 and 44 look alike to it. The slow pair separates them. The next block makes the table as a function, prints the numbers of the hand calculations, and checks the clock at the paper’s size.

In [ ]:
import numpy as np

def pe(n, d):
    pos = np.arange(n)[:, None]
    ang = pos / 10000 ** (2 * np.arange(d // 2) / d)
    out = np.zeros((n, d))
    out[:, 0::2], out[:, 1::2] = np.sin(ang), np.cos(ang)
    return out

print(np.round(pe(4, 4), 4))
print(np.round(pe(3, 8)[2], 4))
w = 2 * np.pi * 10000 ** (2 * np.array([0, 128, 255]) / 512)
print(np.round(w, 2), round(2 * np.pi * 10000, 2))

*The lesson shows:*

```text
[[ 0.      1.      0.      1.    ]
 [ 0.8415  0.5403  0.01    1.    ]
 [ 0.9093 -0.4161  0.02    0.9998]
 [ 0.1411 -0.99    0.03    0.9996]]
[ 0.9093 -0.4161  0.1987  0.9801  0.02    0.9998  0.002   1.    ]
[6.280000e+00 6.283200e+02 6.061148e+04] 62831.85
```

The first four lines are the table, and the next line is the row for position 2 at size 8. The wavelengths of pairs 0, 128 and 255 at the paper’s size of 512 are 6.28, 628.32 and 60,611.48, and the limit the paper names, $10000 \cdot 2\pi$, is 62,831.85; the last pair never quite reaches it because $i$ stops at 255. Now the claims: how long a row is, how far apart two rows are at the paper’s size, and the clock that nearly repeats.

In [ ]:
P = pe(1024, 512)
D = np.linalg.norm(P[:, None] - P[None], axis=2) + 1e9 * np.eye(1024)
print(round(np.linalg.norm(P[5]), 4), round(D.min(), 3))
print(np.round(pe(45, 2)[[0, 44]], 4))
print(np.round(pe(45, 4)[[0, 44]], 4))

*The lesson shows:*

```text
16.0 3.714
[[0.     1.    ]
 [0.0177 0.9998]]
[[0.     1.     0.     1.    ]
 [0.0177 0.9998 0.4259 0.9048]]
```

### What the paper hoped for

The paper chose waves for a reason it states as a hypothesis, not a result: for any fixed offset $k$, the row for position $pos + k$ should be a linear function of the row for $pos$, so that the model can learn to attend by relative position. You can check the claim in the language of lesson 03.5. In each pair, going $k$ steps on turns the point $(\sin x, \cos x)$ to $(\sin(x + t), \cos(x + t))$ with $t = k$ divided by the pair’s divisor, which is a rotation by $t$. For the whole row that is one table with a $2 \times 2$ rotation for each pair, and it depends on $k$ but not on $pos$.

In [ ]:
def turn(k, d=4):
    M = np.zeros((d, d))
    for i in range(d // 2):
        t = k / 10000 ** (2 * i / d)
        c, s = np.cos(t), np.sin(t)
        M[2 * i:2 * i + 2, 2 * i:2 * i + 2] = [[c, -s], [s, c]]
    return M

P = pe(20, 4)
print([np.allclose(P[p] @ turn(1), P[p + 1]) for p in (2, 4, 9)])
print(round(P[2] @ P[5], 4), round(P[7] @ P[10], 4), round(P[2] @ P[3], 4))

*The lesson shows:*

```text
[True, True, True]
0.0096 0.0096 1.5403
```

## 12.4 Attention scores: scaled dot products

Each token’s query is dotted with every token’s key, and the result is scaled so its spread stays near one. It uses the dot product from Module 02 and the random-vector spread from Modules 07 and 11.

[Read the lesson](https://aicourses.dev/mathematics/inside-a-transformer/attention-scores)

### In NumPy

The first block does the hand example both ways: a loop with two `for`s and the sum over the four entries, and then the matrix product. The scores divided by 2 are the table of the next lesson.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
X = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [1, 1, 0, 0]], float)
Wq = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 0, 0], [0, 0, 0, 1]], float)
Wk = np.array([[0, 1, 0, 1], [1, 0, 1, 0], [0, 0, 1, 0], [1, 0, 0, 0]], float)
Q, K = X @ Wq, X @ Wk
S = np.zeros((3, 3))
for i in range(3):
    for j in range(3):
        S[i, j] = sum(Q[i, t] * K[j, t] for t in range(4))
print(S)
print(np.allclose(S, Q @ K.T))
print(Q @ K.T / np.sqrt(4))

*The lesson shows:*

```text
[[1. 5. 3.]
 [3. 0. 3.]
 [3. 3. 4.]]
True
[[0.5 2.5 1.5]
 [1.5 0.  1.5]
 [1.5 1.5 2. ]]
```

Now the footnote: the spread of the dot product at three key sizes, from 10,000 random pairs each, then the softmax of rows of eight scores with spread 1 and 8, and a nudge check of the slope of a share.

In [ ]:
for d in (4, 16, 64):
    q, k = rng.normal(size=(10000, d)), rng.normal(size=(10000, d))
    s = (q * k).sum(axis=1)
    print(d, round(s.std(), 2), round((s / np.sqrt(d)).std(), 2))

def softmax(S):
    e = np.exp(S - S.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)
for spread in (1, 8):
    top = softmax(rng.normal(0, spread, (10000, 8))).max(axis=1)
    print(spread, round(top.mean(), 3), round((top > 0.99).mean(), 3))

*The lesson shows:*

```text
4 2.0 1.0
16 4.02 1.0
64 8.03 1.0
1 0.361 0.0
8 0.877 0.378
```

The first three lines are simulated, so their last digits belong to this seed: they land close to 2, 4 and 8, and close to 1 after the division. The slope claim can be checked by nudging, as in lesson 04.9:

In [ ]:
h = 1e-5
for x in (0.0, np.log(99)):
    s = np.array([x, 0.0])
    up, dn = softmax(s + [h, 0])[0], softmax(s - [h, 0])[0]
    print(round(softmax(s)[0], 4), round((up - dn) / (2 * h), 4))

*The lesson shows:*

```text
0.5 0.25
0.99 0.0099
```

## 12.5 Softmax, masks and a weighted average

Softmax turns each row of scores into shares, a mask hides the future, and the output is a weighted average of values. You work three tokens by hand and see attention as a calculation you can check.

[Read the lesson](https://aicourses.dev/mathematics/inside-a-transformer/softmax-and-masks)

### In NumPy

First attention with no mask, the equation straight from the paper, using the stable softmax:

In [ ]:
import numpy as np

def softmax(S):
    e = np.exp(S - S.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

X = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [1, 1, 0, 0]], float)
Wq = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 0, 0], [0, 0, 0, 1]], float)
Wk = np.array([[0, 1, 0, 1], [1, 0, 1, 0], [0, 0, 1, 0], [1, 0, 0, 0]], float)
Wv = np.array([[1, 0, 0, 0], [0, 2, 0, 0], [0, 0, 1, 0], [0, 0, 0, 2]], float)
Q, K, V = X @ Wq, X @ Wk, X @ Wv
S = Q @ K.T / np.sqrt(4)
A = softmax(S)
print(np.round(A, 4))
print(np.round(A @ V, 4))

*The lesson shows:*

```text
[[0.09   0.6652 0.2447]
 [0.4498 0.1004 0.4498]
 [0.2741 0.2741 0.4519]]
[[0.3348 1.8199 0.09   1.3305]
 [0.8996 1.1004 0.4498 0.2007]
 [0.7259 1.4519 0.2741 0.5481]]
```

Row 2 of the output matches the hand sum above. Now the mask, with `np.triu` (the part of a table above its diagonal) choosing what to hide:

In [ ]:
mask = np.triu(np.ones((3, 3), bool), k=1)
Sm = np.where(mask, -np.inf, S)
Am = softmax(Sm)
print(Sm)
print(np.round(Am, 4), Am.sum(axis=1))
print(np.round(Am @ V, 4))
print(np.allclose(softmax(np.where(mask, -1e10, S)), Am))

*The lesson shows:*

```text
[[ 0.5 -inf -inf]
 [ 1.5  0.  -inf]
 [ 1.5  1.5  2. ]]
[[1.     0.     0.    ]
 [0.8176 0.1824 0.    ]
 [0.2741 0.2741 0.4519]] [1. 1. 1.]
[[1.     0.     1.     0.    ]
 [0.8176 0.3649 0.8176 0.3649]
 [0.7259 1.4519 0.2741 0.5481]]
True
```

## 12.6 Many heads, and shapes with einsum

Several small attentions run side by side and are joined and mixed. You track every shape and write the whole thing once with einsum.

[Read the lesson](https://aicourses.dev/mathematics/inside-a-transformer/many-heads)

### In NumPy

The loop version first. Each head takes its two columns of the queries, keys and values (the same as using its slice of each matrix), runs the attention of lesson 12.5 without a mask, and the heads are joined and mixed:

In [ ]:
import numpy as np

def softmax(S):
    e = np.exp(S - S.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

X = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [1, 1, 0, 0]], float)
Wq = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 0, 0], [0, 0, 0, 1]], float)
Wk = np.array([[0, 1, 0, 1], [1, 0, 1, 0], [0, 0, 1, 0], [1, 0, 0, 0]], float)
Wv = np.array([[1, 0, 0, 0], [0, 2, 0, 0], [0, 0, 1, 0], [0, 0, 0, 2]], float)
Wo = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 0, 1], [0, 1, 1, 0]], float)
h, dk = 2, 2
Q, K, V = X @ Wq, X @ Wk, X @ Wv

In [ ]:
def attend(Q, K, V):
    return softmax(Q @ K.T / np.sqrt(K.shape[-1])) @ V

heads = []
for i in range(h):
    cols = slice(i * dk, (i + 1) * dk)           # this head's columns
    heads.append(attend(Q[:, cols], K[:, cols], V[:, cols]))
print(heads[0].shape, np.round(heads[0][0], 4))
cat = np.concatenate(heads, axis=1)
out = cat @ Wo
print(cat.shape, out.shape, np.round(out[0], 4))

*The lesson shows:*

```text
(3, 2) [0.2321 1.9092]
(3, 4) (3, 4) [0.5654 2.5759 0.8987 2.2426]
```

The shapes are those of the table, and row 0 of the result is the number found by hand, 0.5654. Now einsum, first on small things you can check by head, then on the whole multi-head attention:

In [ ]:
a, b = np.array([1.0, 2, 3]), np.array([4.0, 5, 6])
print(np.einsum("i,i->", a, b), a @ b)
print(np.einsum("i,j->ij", a, b).shape)
print(np.allclose(np.einsum("ij,jk->ik", X, Wq), X @ Wq))

*The lesson shows:*

```text
32.0 32.0
(3, 3)
True
```

The dot product is 32, as $1 \times 4 + 2 \times 5 + 3 \times 6$, and the two others give a table of products and the matrix product. Now the heads, with the matrices cut into heads by one `reshape`: each column of $(4, 4)$ becomes (head, place in head).

In [ ]:
Wq_h = Wq.reshape(4, h, dk).transpose(1, 0, 2)   # (h, d_model, d_k)
Wk_h = Wk.reshape(4, h, dk).transpose(1, 0, 2)
Wv_h = Wv.reshape(4, h, dk).transpose(1, 0, 2)
Qh = np.einsum("nm,hmd->hnd", X, Wq_h)
Kh = np.einsum("nm,hmd->hnd", X, Wk_h)
Vh = np.einsum("nm,hmd->hnd", X, Wv_h)
S = np.einsum("hnd,htd->hnt", Qh, Kh) / np.sqrt(dk)
Hh = np.einsum("hnt,htd->hnd", softmax(S), Vh)
out2 = np.einsum("hnd,hdm->nm", Hh, Wo.reshape(h, dk, 4))
print(Qh.shape, S.shape, Hh.shape, out2.shape)
print(np.allclose(out2, out))

*The lesson shows:*

```text
(2, 3, 2) (2, 3, 3) (2, 3, 2) (3, 4)
True
```

The loop and the einsum give the same table. Last, the same code at the paper’s own sizes, with random matrices (their values are not the point, the shapes are), for 5 tokens:

In [ ]:
rng = np.random.default_rng(0)
n, d_model, H, d_k = 5, 512, 8, 64
X5 = rng.normal(size=(n, d_model))
Wq5 = rng.normal(size=(H, d_model, d_k)) * d_model ** -0.5
Q5 = np.einsum("nm,hmd->hnd", X5, Wq5)
print(Q5.shape, 8 * 512 * 64, 3 * 8 * 512 * 64 + 512 * 512, 4 * 512 * 512)

*The lesson shows:*

```text
(8, 5, 64) 262144 1048576 1048576
```

## 12.7 Residual connections and layer normalisation

Adding the input back gives the slope a straight path, and layer normalisation keeps each vector’s scale steady. These two small devices are what let a deep stack train.

[Read the lesson](https://aicourses.dev/mathematics/inside-a-transformer/residuals-and-normalisation)

### In NumPy

The row $(1, 2, 3, 6)$ with a loop, as in lesson 01.4, then one line each, and then with a gain and a shift, and with the row scaled:

In [ ]:
import numpy as np

x = np.array([1.0, 2.0, 3.0, 6.0])
mu = 0
for a in x:
    mu = mu + a / 4
ss = 0
for a in x:
    ss = ss + (a - mu) ** 2
sigma = (ss / 4) ** 0.5
print(mu, round(sigma, 4), np.round((x - mu) / sigma, 4))
z = (x - x.mean()) / x.std()
print(np.round(z, 4), round(z.mean(), 4), round(z.std(), 4))

*The lesson shows:*

```text
3.0 1.8708 [-1.069  -0.5345  0.      1.6036]
[-1.069  -0.5345  0.      1.6036] 0.0 1.0
```

In [ ]:
g = np.array([2.0, 1.0, 1.0, 1.0])
b = np.array([0.0, 0.0, 0.0, 10.0])
print(np.round(g * z + b, 4))
for scale in (1, 10, 100):
    y = scale * x
    print(scale, np.round((y - y.mean()) / y.std(), 4))

*The lesson shows:*

```text
[-2.1381 -0.5345  0.     11.6036]
1 [-1.069  -0.5345  0.      1.6036]
10 [-1.069  -0.5345  0.      1.6036]
100 [-1.069  -0.5345  0.      1.6036]
```

Real code adds a tiny number $\epsilon$ to the variance before the square root, so that a row of identical numbers, whose spread is 0, does not divide by zero. The equations of Ba and others have none; OpenAI’s published code for GPT-2 uses $10^{-5}$. And a batch of tokens is normalised one row at a time with `axis=1`:

In [ ]:
def ln(x, g, b, eps=1e-5):
    return g * (x - x.mean()) / np.sqrt(x.var() + eps) + b

print(np.round(ln(x, 1, 0), 4), np.round(ln(x, 1, 0, eps=0.0), 4))
X = np.array([x, [0.0, 0.0, 0.0, 4.0]])
Xn = (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)
print(np.round(Xn, 4))

*The lesson shows:*

```text
[-1.069  -0.5345  0.      1.6036] [-1.069  -0.5345  0.      1.6036]
[[-1.069  -0.5345  0.      1.6036]
 [-0.5774 -0.5774 -0.5774  1.7321]]
```

With $\epsilon = 10^{-5}$ the first four decimals do not move. Now the slopes: twelve steps plain and with the input added back, and the nudge check of $x + 0.05x^2$:

In [ ]:
slope_f = 0.05
plain, resid = 1.0, 1.0
for layer in range(12):
    plain *= slope_f
    resid *= 1 + slope_f
print(plain, round(resid, 4))
full = lambda t: t + 0.05 * t ** 2
h = 1e-5
for t0 in (2.0, 3.0):
    d = (full(t0 + h) - full(t0 - h)) / (2 * h)
    print(t0, round(d, 4), round(1 + 0.1 * t0, 4))

*The lesson shows:*

```text
2.4414062500000014e-16 1.7959
2.0 1.2 1.2
3.0 1.3 1.3
```

## 12.8 The feed-forward layer and the output probabilities

For each token alone, the feed-forward layer widens, bends and narrows, and then one matrix and softmax give a chance for every token. It completes the forward pass.

[Read the lesson](https://aicourses.dev/mathematics/inside-a-transformer/feed-forward-and-output)

### In NumPy

The feed-forward layer, for the two tokens, and the check that a row’s output depends on its own row only:

In [ ]:
import numpy as np

W1 = np.array([[1, 0, -1], [2, 1, 1]])           # (2, 3): the paper's form
b1 = np.array([0.5, 0, -1])
W2 = np.array([[1, 0], [0, 1], [1, 1]])          # (3, 2)
b2 = np.array([0, 0.5])
ffn = lambda X: np.maximum(0, X @ W1 + b1) @ W2 + b2
X = np.array([[2, 1], [0, 3]])
print(ffn(X))
print(ffn(X[1:]), ffn(np.array([[5, -4], [0, 3]]))[1])
print(W1.size + b1.size + W2.size + b2.size)
print(512 * 2048 + 2048 + 2048 * 512 + 512)

*The lesson shows:*

```text
[[4.5 1.5]
 [8.5 5.5]]
[[8.5 5.5]] [8.5 5.5]
17
2099712
```

Token 1’s output is (8.5, 5.5) alone, and also with a different token 0, $(5, -4)$. Now the output step:

In [ ]:
def softmax(S):
    e = np.exp(S - S.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

E = np.array([[1, 0], [0, 1], [1, 1]])           # 3 tokens, each a row
h = np.array([1, 2])                             # final row of a position
logits = h @ E.T
q = softmax(logits)
print(logits, np.round(q, 4))
print(round(-np.log(q[1]), 4), round(np.log(3), 4))

*The lesson shows:*

```text
[1 2 3] [0.09   0.2447 0.6652]
1.4076 1.0986
```

## 12.9 Counting parameters

Add up every matrix in the paper’s base model and in GPT-2 small, and compare your total with what the papers state. It tests every shape you have tracked.

[Read the lesson](https://aicourses.dev/mathematics/inside-a-transformer/counting-parameters)

### In NumPy

The base model’s matrices, from the sizes above, then the small items and the total:

In [ ]:
V, d, d_ff = 37000, 512, 2048
attn, ffn = 4 * d * d, 2 * d * d_ff               # matrices only
enc, dec = attn + ffn, 2 * attn + ffn             # decoder has two attentions
emb = V * d                                       # one shared table
core = emb + 6 * enc + 6 * dec
print(attn, ffn, enc, dec, emb)
print(core)

*The lesson shows:*

```text
1048576 2097152 3145728 4194304 18944000
62984192
```

In [ ]:
ffn_bias = 6 * 2 * (d_ff + d)                     # b1 and b2, 12 feed-forwards
attn_bias = (6 + 12) * 4 * d                      # 18 attentions
norms = (6 * 2 + 6 * 3) * 2 * d                   # 30 norms, gain and shift
print(ffn_bias, attn_bias, norms)
total = core + ffn_bias + attn_bias + norms
print(total, round(total / 1e6, 2), round(65e6 - total))
print(round((65e6 - total) / d))

*The lesson shows:*

```text
30720 36864 30720
63082496 63.08 1917504
3745
```

Then one function for a model like GPT-2, tried on the tiny model of the guess and on GPT-2 small, and the paper’s feed-forward rows:

In [ ]:
def count(V, d, d_ff, L, n_ctx):         # learned positions, tied output
    emb = V * d + n_ctx * d
    blk = 4 * d * d + 4 * d + 2 * d * d_ff + d_ff + d
    blk += 4 * d                         # two norms
    return emb + L * blk + 2 * d         # and the final norm

print(count(100, 8, 32, 2, 16))
print(count(50257, 768, 3072, 12, 1024))

def base(d_ff, d=512, V=37000):
    attn, ffn = 4 * d * d, 2 * d * d_ff
    return V * d + 6 * (attn + ffn) + 6 * (2 * attn + ffn)
for d_ff, paper in [(1024, 53), (2048, 65), (4096, 90)]:
    print(d_ff, paper, round(base(d_ff) / 1e6, 2))

*The lesson shows:*

```text
2688
124439808
1024 53 50.4
2048 65 62.98
4096 90 88.15
```

## 12.10 How the gradient flows back, and how it is trained

The gradient starts as the chances minus the answer and flows back through each residual, and Adam trains the result with the paper’s warmup formula. It joins this module to Modules 05, 06 and 09.

[Read the lesson](https://aicourses.dev/mathematics/inside-a-transformer/gradient-flow-and-training)

### In NumPy

The gradient at the top, the residual and the nudge check:

In [ ]:
import numpy as np

def softmax(S):
    e = np.exp(S - S.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

E = np.array([[1.0, 0], [0, 1], [1, 1]])         # 3 tokens, 2 numbers each
A = np.array([[0.1, 0.0], [0.2, 0.1]])           # a made-up sub-layer: x @ A
t = np.array([0.0, 1.0, 0.0])                    # the true token: the second
x = np.array([1.0, 2.0])
y = x + x @ A
q = softmax(y @ E.T)
dy = (q - t) @ E                                 # chances minus answer, then E
dx = dy + dy @ A.T                               # pass down, plus a correction
print(np.round(q - t, 4), np.round(dy, 4), np.round(dx, 4))

*The lesson shows:*

```text
[ 0.0831 -0.8327  0.7497] [ 0.8327 -0.0831] [0.916  0.0752]
```

In [ ]:
loss = lambda x: -np.log(softmax((x + x @ A) @ E.T)[1])
h = 1e-5
num = [(loss(x + h * e) - loss(x - h * e)) / (2 * h) for e in np.eye(2)]
print(np.round(num, 4))

*The lesson shows:*

```text
[0.916  0.0752]
```

The straight path is `dy` and the correction `dy @ A.T`, and the nudge check agrees with the sum. Now the paper’s schedule and Adam’s first step:

In [ ]:
d_model, warmup = 512, 4000
for step in [1, 1000, 2000, 4000, 8000, 16000, 100000]:
    lr = d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)
    print(step, f"{lr:.2e}")

b1, b2, eps = 0.9, 0.98, 1e-9
lr1 = d_model ** -0.5 * 1 * warmup ** -1.5       # the learning rate at step 1
for g in [0.5, 500.0, 0.0005]:
    m_hat, v_hat = g, g * g                      # step 1, corrected
    print(g, f"{lr1 * m_hat / (np.sqrt(v_hat) + eps):.3e}")

*The lesson shows:*

```text
1 1.75e-07
1000 1.75e-04
2000 3.49e-04
4000 6.99e-04
8000 4.94e-04
16000 3.49e-04
100000 1.40e-04
0.5 1.747e-07
500.0 1.747e-07
0.0005 1.747e-07
```

And the smoothed target, with a model that is very sure and a model that gives the smooth table itself:

In [ ]:
K, eps_ls = 46, 0.1
sharp = np.zeros(K); sharp[5] = 1.0
smooth = (1 - eps_ls) * sharp + eps_ls / K
print(round(smooth[5], 4), round(smooth[0], 5), round(smooth.sum(), 4))
sure = np.full(K, 0.01 / 45); sure[5] = 0.99     # a very confident model
ce = lambda target, q: -(target * np.log(q)).sum()
for name, q in [("sure", sure), ("smooth", smooth)]:
    print(name, round(ce(sharp, q), 4), round(ce(smooth, q), 4))

*The lesson shows:*

```text
0.9022 0.00217 1.0
sure 0.0101 0.832
smooth 0.1029 0.6927
```

## 12.11 What changed since 2017

This optional, dated lesson puts what changed since 2017 in one table: where the normalisation sits, a simpler normalisation, rotary positions and a different bend. Each change comes with its paper.

[Read the lesson](https://aicourses.dev/mathematics/inside-a-transformer/since-2017)

### In NumPy

Layer and RMS normalisation of the same row, and on a row with mean 0:

In [ ]:
import numpy as np

x = np.array([1.0, 2.0, 3.0, 6.0])
z = np.array([-2.0, -1.0, 0.0, 3.0])
layer = lambda a: (a - a.mean()) / a.std()
rms = lambda a: a / np.sqrt((a ** 2).mean())
print(np.round(layer(x), 4), np.round(rms(x), 4))
print(np.round(layer(z), 4), np.round(rms(z), 4))
print(round(np.sqrt((x ** 2).mean()), 4))

*The lesson shows:*

```text
[-1.069  -0.5345  0.      1.6036] [0.2828 0.5657 0.8485 1.6971]
[-1.069  -0.5345  0.      1.6036] [-1.069  -0.5345  0.      1.6036]
3.5355
```

The bends, and the tanh formula that GPT-2’s published code computes in place of the exact GELU:

In [ ]:
import math

for v in (-1.0, 0.0, 1.0, 2.0):
    gelu = v * 0.5 * (1 + math.erf(v / 2 ** 0.5))
    k = math.sqrt(2 / math.pi) * (v + 0.044715 * v ** 3)
    tanh = 0.5 * v * (1 + math.tanh(k))
    silu = v / (1 + math.exp(-v))
    print(v, max(0, v), round(gelu, 4), round(tanh, 4), round(silu, 4))
print(3 * 768 * 2048, 2 * 768 * 3072)

*The lesson shows:*

```text
-1.0 0 -0.1587 -0.1588 -0.2689
0.0 0 0.0 0.0 0.0
1.0 1.0 0.8413 0.8412 0.7311
2.0 2.0 1.9545 1.9546 1.7616
4718592 4718592
```

## 12.12 Project: one attention block by hand

You write one attention block in NumPy, from eight characters of text to output chances and a loss, and count every parameter. It is the end of the road and the proof that you can read the paper.

[Read the lesson](https://aicourses.dev/mathematics/inside-a-transformer/project-attention-by-hand)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Step 1: the text and the table

Load the course’s text as in Module 09 and cut nine characters from it: eight to read and the eight that follow them, which are the answers. Use a row size of 16 and a feed-forward middle of 64. The table $E$ for the 46 characters gets random numbers of spread $1/\sqrt{16}$, a choice of this project and not the paper’s, and each character’s row is multiplied by $\sqrt{16} = 4$ (section 3.4) before the position waves of lesson 12.3 are added (each position row has length $\sqrt{8} = 2.828$).

In [ ]:
import urllib.request
import numpy as np

rng = np.random.default_rng(0)
DATA = "https://raw.githubusercontent.com/aicoursesdev/mathematics-for-ai/main/data/"
urllib.request.urlretrieve(DATA + "alice.txt", "alice.txt")
text = " ".join(open("alice.txt", encoding="utf-8").read().lower().split())
chars = sorted(set(text))
V, d, d_ff, n = len(chars), 16, 64, 8
ids = np.array([chars.index(c) for c in "alice was"])    # nine characters
x_ids, y_ids = ids[:-1], ids[1:]                         # eight in, eight to guess
print(V, x_ids, y_ids)

*The lesson shows:*

```text
46 [14 25 22 16 18  0 36 14] [25 22 16 18  0 36 14 32]
```

In [ ]:
E = rng.normal(0, d ** -0.5, (V, d))
ang = np.arange(n)[:, None] / 10000 ** (2 * np.arange(d // 2) / d)
PE = np.zeros((n, d))
PE[:, 0::2], PE[:, 1::2] = np.sin(ang), np.cos(ang)
X = E[x_ids] * np.sqrt(d) + PE
print(X.shape, np.round(X[0, :4], 3), np.round(np.linalg.norm(PE[3]), 3))

*The lesson shows:*

```text
(8, 16) [-0.5    2.36   1.002  0.848] 2.828
```

### Step 2: one masked head

Make four matrices of 16 by 16, for queries, keys, values and the final mix. Score every character against every one (divided by $\sqrt{16}$), set the scores above the diagonal to minus infinity, take a stable softmax of each row, and mix the values. With a single head, the matrix $W^O$ is one more matrix.

In [ ]:
def softmax(S):
    e = np.exp(S - S.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

W = {k: rng.normal(0, d ** -0.5, (d, d)) for k in ["Wq", "Wk", "Wv", "Wo"]}
S = (X @ W["Wq"]) @ (X @ W["Wk"]).T / np.sqrt(d)
S = np.where(np.triu(np.ones((n, n), bool), k=1), -np.inf, S)
A = softmax(S)
H = A @ (X @ W["Wv"]) @ W["Wo"]
print(np.round(A, 2))
print(A.sum(axis=1), H.shape)

*The lesson shows:*

```text
[[1.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.9  0.1  0.   0.   0.   0.   0.   0.  ]
 [0.09 0.9  0.01 0.   0.   0.   0.   0.  ]
 [0.68 0.03 0.27 0.02 0.   0.   0.   0.  ]
 [0.02 0.38 0.08 0.51 0.01 0.   0.   0.   ]
 [0.06 0.42 0.01 0.48 0.   0.03 0.   0.  ]
 [0.11 0.31 0.04 0.51 0.   0.03 0.01 0.  ]
 [0.27 0.28 0.03 0.23 0.   0.05 0.01 0.13]]
[1. 1. 1. 1. 1. 1. 1. 1.] (8, 16)
```

### Step 3: add, normalise, widen, narrow

Add the block’s input back to its output and normalise (lesson 12.7, the paper’s order), then run the feed-forward layer of lesson 12.8 on each row, add back and normalise again. Gains start at 1 and shifts at 0.

In [ ]:
def layer_norm(Z, g, b, eps=1e-5):
    mu, var = Z.mean(axis=-1, keepdims=True), Z.var(axis=-1, keepdims=True)
    return g * (Z - mu) / np.sqrt(var + eps) + b

g1, b1n, g2, b2n = np.ones(d), np.zeros(d), np.ones(d), np.zeros(d)
W1, c1 = rng.normal(0, d ** -0.5, (d, d_ff)), np.zeros(d_ff)
W2, c2 = rng.normal(0, d_ff ** -0.5, (d_ff, d)), np.zeros(d)
Y = layer_norm(X + H, g1, b1n)
F = np.maximum(0, Y @ W1 + c1) @ W2 + c2
Z = layer_norm(Y + F, g2, b2n)
print(Z.shape, np.allclose(Z.mean(axis=1), 0), np.round(Z.std(axis=1), 3))

*The lesson shows:*

```text
(8, 16) True [1. 1. 1. 1. 1. 1. 1. 1.]
```

### Step 4: chances and the loss

The last step reuses the table $E$, as in the paper: the scores are $Z E^\top$, and a softmax of each row gives 46 chances. The loss is the average of minus the log of the chance given to the true next character (lesson 09.4).

In [ ]:
logits = Z @ E.T                             # the same table E, as the paper
Q = softmax(logits)
loss = -np.log(Q[np.arange(n), y_ids]).mean()
print(Q.shape, np.round(Q.sum(axis=1), 3))
print(round(loss, 3), round(np.log(V), 3))

*The lesson shows:*

```text
(8, 46) [1. 1. 1. 1. 1. 1. 1. 1.]
4.535 3.829
```

Eight rows of 46 chances, each adding to 1. The loss is 4.535 and a uniform guess is 3.829, so the block is worse than guessing by 0.706. Why? Two things you can measure:

In [ ]:
sd = logits.std()
print(round(sd, 3), round(np.log(V) + sd ** 2 / 2, 3))
r = np.arange(n)
print(round(logits[r, x_ids].mean(), 3), round(logits.mean(), 3))
print(round(logits[r, y_ids].mean(), 3))

*The lesson shows:*

```text
0.905 4.238
2.105 0.045
-0.224
```

### Step 5: count every number

Put every array in one dictionary and add the sizes. The position waves hold nothing, and the output step reuses $E$.

In [ ]:
params = dict(E=E, **W, g1=g1, b1n=b1n, W1=W1, c1=c1)
params.update(W2=W2, c2=c2, g2=g2, b2n=b2n)
for k, v in params.items():
    print(k, v.shape, v.size)
print(sum(v.size for v in params.values()))

*The lesson shows:*

```text
E (46, 16) 736
Wq (16, 16) 256
Wk (16, 16) 256
Wv (16, 16) 256
Wo (16, 16) 256
g1 (16,) 16
b1n (16,) 16
W1 (16, 64) 1024
c1 (64,) 64
W2 (64, 16) 1024
c2 (16,) 16
g2 (16,) 16
b2n (16,) 16
3952
```

### Stretch goals

One gradient step on the output table: the gradient at the scores is $(Q - T)/n$, with $T$ the table of answers (lesson 09.8), and the gradient for $E$ through the output step is that table’s transpose times $Z$. This updates only the output path of $E$, and the input path is left as it is, so it is part of the gradient, not all. A nudge check confirms two entries.

In [ ]:
T = np.eye(V)[y_ids]
dE = ((Q - T) / n).T @ Z
E2 = E - 1.0 * dE
Q2 = softmax(Z @ E2.T)
print(round(-np.log(Q2[np.arange(n), y_ids]).mean(), 3))
f = lambda M: -np.log(softmax(Z @ M.T)[np.arange(n), y_ids]).mean()
for i, j in [(0, 0), (14, 3)]:
    D = np.zeros_like(E); D[i, j] = 1e-5
    print(i, j, round(dE[i, j], 5), round((f(E + D) - f(E - D)) / 2e-5, 5))

*The lesson shows:*

```text
2.869
0 0 0.1724 0.1724
14 3 0.09177 0.09177
```